# GX_04
# Differential Interference Contrast Microscopy

In this graded exercise, you will be responsible for implementing a differential interference contrast (DIC) microscope using the angular spectrum method.

A DIC microscope works on a simple principle where an incoming beam of linearly polarized light (at 45°) is split into its vertical and horizontal polarization states with a prism that imparts a different tilt to each polarization. The two beams is then collimated with a lens, now having accumulated a slight offset relative to one another. The beams then pass through the sample in question, and are brought back to the same physical location with another lens and a second prism to undo the tilts. A waveplate may also be used to create a relative global phase shift between the two beams before they are recombined. This creates an output pattern that is the sum of two slightly offset copies of the object, which leads to constructive interference on some object edges and destructive interference on others. Our brain naturally interprets this effect as shadows, and as such sees the resulting image as a height map.

<div style="text-align: center;">
    <br>
    <img src="https://upload.wikimedia.org/wikipedia/commons/thumb/4/46/DIC_Light_Path.png/1280px-DIC_Light_Path.png">
    <p>
        More information on DIC microscopy may be found here: 
        <a href="url">https://en.wikipedia.org/wiki/Differential_interference_contrast_microscopy</a>
    </p>
    <br>
</div>

In [ ]:
# - No modification necessary

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

from skimage import data, transform

# ============================================================
# Simulation constants
# ============================================================
f = 100e-3
wavelength = 633e-9
k = 2*np.pi/wavelength

dx = 15e-6
N = 512

x = (np.arange(N) - N//2) * dx
y = (np.arange(N) - N//2) * dx
X, Y = np.meshgrid(x, y)

kx = 2 * np.pi * np.fft.fftfreq(N, dx)
ky = 2 * np.pi * np.fft.fftfreq(N, dx)
KX, KY = np.meshgrid(kx, ky)
KZ = np.sqrt(np.maximum(0, k**2 - KX**2 - KY**2))

# ============================================================
# Sample phase image
# ============================================================
padding = N*3//4

sample_img = data.coins()
sample_img = transform.resize(sample_img, (N - padding, N - padding))
sample_img = sample_img / sample_img.max()
sample_img = sample_img * np.pi / 2
sample_field = np.exp(1j * sample_img)
sample = np.ones((N, N), dtype=complex)
start = padding // 2
end = start + sample_field.shape[0]
sample[start:end, start:end] = sample_field

# ============================================================
# Input Gaussian beam
# ============================================================

w0 = (N-padding)*dx
gaussian = np.exp(-(X**2 + Y**2) / (w0**2))
field = np.ones((N, N), dtype=complex) * gaussian

# ============================================================
# Helper Functions
# ============================================================

def angular_spectrum_propagate(U_in, z):
    U_f = np.fft.fft2(U_in)
    U_f = U_f * np.exp(1j * KZ * z)
    U_out = np.fft.ifft2(U_f)

    return U_out


# Part 1
Note the following information for your implementation:
1) This microscope works as a series of cascaded image and Fourier planes. This means that every component is spaced out by one focal length. For simplicity, you should only use one focal length throughout the whole system (which has already been provided).
2) The prisms in this system are located at Fourier planes, **not** image planes. This means that your incoming beam, which will start at an image plane, must be converted to Fourier space before splitting with the prism. Similarly, your output beam must be returned to the image plane after the second prism.
3) Different polarizations of light do not interfere. For the sake of our simulation, this means that we want to treat them as completely independent beams. You should not attempt to model the polarization of the light, and instead should manually implement two different propagation paths between the prisms.
4) Do not attempt to model the prism in any physical sense. Instead, just assume that the prism imparts a tilt of the provided magnitude along the provided axis to half of the beam's intensity, and an opposite tilt to the other half.

The phase sample that you will be imaging has already been provided for you, as has the input beam to your system. Keeping in mind the above information, implement the following in the function image_with_DIC to create your microscope:
1) Conjugate the input beam to Fourier space
2) Split into two beams with opposite tilts
3) Conjugate back into image space to produce two copies of the input with a spatial offset
4) Apply the sample to each beam
5) Conjugate the resulting beams into Fourier space
6) Undo the beam tilts and recombine with a relative phase shift
7) Conjugate to the image plane and measure the intensity

In [ ]:
def image_with_DIC(axis, amplitude, phase_bias):
    """
    Compute an intensity image using a DIC-like interferometric setup.
    Note that you may wish to create some helper functions for commonly repeated actions.
    
    Parameters
    ----------
    axis : direction along which the phase ramps are applied
    amplitude : magnitude of the phase ramp applied to the fields
    phase_bias : relative phase shift between the two interfering components
    
    Returns
    -------
    I_out : output intensity image (2D array)
    """
    
    raise NotImplementedError


In [ ]:
# - No modification necessary -

def visualize_DIC_output(axis, amplitude, phase_bias):
    I_out = image_with_DIC(axis, amplitude, phase_bias)
    
    plt.imshow(I_out[start-8:end+8, start-8:end+8], cmap="gray_r")
    plt.colorbar()
    plt.show()

interact(
    visualize_DIC_output,
    axis=FloatSlider(min=0, max=2*np.pi, step=np.pi/32, value=0),
    amplitude=FloatSlider(min=0, max=1e-4, step=0.5e-5, value=5e-5, readout_format='.6f'),
    phase_bias=FloatSlider(min=0, max=2*np.pi, step=np.pi/32, value=np.pi)
)

# Part 2
After you have completed your information of the DIC microscope, use the interactive plot to experiment with the different parameters of the microscope and discuss the following questions:
1) Can you describe the output of the system that you are seeing? How would you use the axis and amplitude of the prism's deflections as a user of the microscope to help you investigate your sample?
2) What role does the phase bias seem to play in the image that you are getting out? What bias seems to yield the nicest looking images? Why?

## Discussion
TODO

# Bonus
Both this and the Zernike phase contrast microscope employ a technique where the probe beam is interfered with itself to generate intensity contrast from phase objects. Can you discuss the similarities and differences between these two techniques and why you might use one or the other?

## Discussion
TODO